In [40]:
"""
Тестирование алгоритмов поиска оптимальных путей
"""
import osmnx as ox
import pandas as pd
import networkx as nx
import multiprocessing as mp
import concurrent.futures
from tqdm.auto import tqdm
from modules import pickle_union as pu
from modules import context_timer as ct

import ride_pfa.clustering as cls
import ride_pfa.path_finding as pfa
import ride_pfa.centroid_graph.centroids_graph_builder as cgb

In [42]:
res_time = {}  # Определим пустой словарь для хранения времени выполнения
for key in ["AStar_nx", "Dijkstra_nx", "AStar_ride", "Dijkstra_ride", "BiDijkstra_ride", "AStar_ride_sub", "Dijkstra_ride_sub", "OSMNX"]:
    res_time[key] = []

In [43]:
def task(index):
    row = test_df.iloc[index]
    start_node = row["node_1"]
    end_node = row["node_2"]

    ### A* (networkx)
    with ct.timer() as elapsed_time:
        nx.astar_path(g, start_node, end_node, heuristic=heuristic, weight='weight')
    res_time['AStar_nx'].append(f"{index}:{elapsed_time()}")

    ### Дийкстра (networkx)
    with ct.timer() as elapsed_time:
        nx.dijkstra_path(g, start_node, end_node, weight='weight')
    res_time['Dijkstra_nx'].append(f"{index}:{elapsed_time()}")

    ### A* (ride)
    with ct.timer() as elapsed_time:
        astar_algo.find_path(start_node, end_node)
    res_time['AStar_ride'].append(f"{index}:{elapsed_time()}")

    ### Дийкстра (ride)
    with ct.timer() as elapsed_time:
        dijkstra_algo.find_path(start_node, end_node)
    res_time['Dijkstra_ride'].append(f"{index}:{elapsed_time()}")

    ### Дийкстра (ride)
    with ct.timer() as elapsed_time:
        bidijkstra_algo.find_path(start_node, end_node)
    res_time['BiDijkstra_ride'].append(f"{index}:{elapsed_time()}")

    ### A* (Центроидный граф > субоптимальный алгоритм)
    with ct.timer() as elapsed_time:
        suboptimal_algorithm_Astar.find_path(start_node, end_node)
    res_time['AStar_ride_sub'].append(f"{index}:{elapsed_time()}")

    ### Дийкстра (Центроидный граф > субоптимальный алгоритм)
    with ct.timer() as elapsed_time:
        suboptimal_algorithm_Dijkstra.find_path(start_node, end_node)
    res_time['Dijkstra_ride_sub'].append(f"{index}:{elapsed_time()}")

    ### OSMNX
    # Рассчитайте кратчайший путь и расстояние между точками
    with ct.timer() as elapsed_time:
        ox.routing.shortest_path(g, start_node, end_node, weight='length')
    res_time['OSMNX'].append(f"{index}:{elapsed_time()}")

    # print(f"{index}  {res_time}", end="\n\n")


In [44]:
"""
Создание графа дорожной сети на основе объединенного графов дорог МО и Москвы
### Graph with 177143 nodes and 228969 edges
"""
save_dir = r"graphs"
with ct.timer() as elapsed_time:
    g = pu.load_and_merge_graphs(pickles_dir=save_dir)
print(f"Построение графа выполнено за:  {elapsed_time():.4f},   |   {g}", end="\n\n")

Построение графа выполнено за:  1.9489,   |   Graph with 177143 nodes and 228969 edges



In [45]:
"""
Считываем датафрейм со сгенерированными нодами
"""
test_df = pd.read_excel("Сгенерированный_датафрейм_1000нод.xlsx", engine="openpyxl")
print(test_df.head(4), end="\n\n")

       node_1       lat1       lon1      node_2       lat2       lon2  MST
0  4571876676  55.713296  36.826896  6221520577  55.712453  36.826315    1
1   364089804  54.854478  37.476844  1241351468  54.953043  37.529576    9
2  2507620827  56.334087  37.029223   327331923  56.329847  37.041418    9
3  1083865729  55.961010  37.801300   702218848  55.959135  37.785477    9



In [46]:
def heuristic(u, v):
    u = g.nodes[u]
    v = g.nodes[v]
    return ((u['x'] - v['x']) ** 2 + (u['y'] - v['y']) ** 2) ** 0.5

In [47]:
"""
RIDE: кластеризация, поиск центроидов
"""
# cms_resolver: Объект для кластеризации
cms_resolver = cls.LouvainCommunityResolver(resolution=1)
# Строим центроидный граф для использования в субоптимальном алгоритме (экономия памяти)
cg = cgb.CentroidGraphBuilder().build(g, cms_resolver)
# Создаем субоптимальный алгоритм поиска пути
suboptimal_algorithm_Dijkstra = pfa.ExtractionPfa(
    g=g,
    upper=pfa.Dijkstra(cg.g),
    down=pfa.Dijkstra(g),
    cluster="cluster"
)
# Создаем субоптимальный алгоритм поиска пути
suboptimal_algorithm_Astar = pfa.ExtractionPfa(
    g=g,
    upper=pfa.AStar(cg.g),
    down=pfa.AStar(g),
    cluster="cluster"
)
dijkstra_algo = pfa.Dijkstra(g)
bidijkstra_algo = pfa.BiDijkstra(g)
astar_algo = pfa.AStar(g)

find edges: 100%|██████████| 710/710 [00:04<00:00, 149.77it/s]


In [49]:
"""
Используем многопоточность для выполнения теста
Каждая итерация будет использовать свой поток
"""
for i in range(10):
    with concurrent.futures.ThreadPoolExecutor() as executor:
        executor.map(task, range(len(test_df)))

    with open(f"file_{i}.txt", "w") as output:
        output.write(str(res_time))

In [ ]:
print(res_time)
with open("file.txt", "w") as output:
    output.write(str(res_time))

# Обработка результатов

In [52]:

# Функция для парсинга значений
def parse_results(results):
    parsed = []
    for algo, values in results.items():
        for entry in values:
            index, time = entry.split(":")
            parsed.append({"index": int(index), "algorithm": algo, "time": float(time)})
    return parsed

In [53]:
# Преобразуем словарь res_time в DataFrame
df = pd.DataFrame(parse_results(res_time))
# Сохраняем DataFrame в Excel-файл
output_file = "results_.xlsx"
df.to_excel(output_file, index=False)
print(f"Результаты сохранены в файле '{output_file}'.")

Результаты сохранены в файле 'results_.xlsx'.


In [55]:
import ast
with open("file_9.txt", "r", encoding="utf-8") as f:
    content = f.read()

In [56]:
data = ast.literal_eval(content)

In [62]:
len(data["AStar_nx"])

10000

In [110]:
df = pd.DataFrame({
    'iteration': 0,
    'index':[],
    'AStar_nx':[],
    'AStar_ride':[],
    'AStar_ride_sub':[],
    'BiDijkstra_ride':[],
    'Dijkstra_nx':[],
    'Dijkstra_ride':[],
    'Dijkstra_ride_sub':[],
    'OSMNX':[]
})

In [111]:
len(df)

0

In [112]:
from tqdm import tqdm
for algo in tqdm(["AStar_nx", "Dijkstra_nx", "AStar_ride", "Dijkstra_ride", "BiDijkstra_ride", "AStar_ride_sub", "Dijkstra_ride_sub", "OSMNX"]):
    for i in range(len(data[algo])):
        index, time = str(data[algo][i]).split(":")
        df.at[i, "index"] = int(index)
        df.at[i, algo] = float(time)

100%|██████████| 8/8 [00:09<00:00,  1.13s/it]


In [100]:
df = df.sort_values("index")

In [118]:
df.rename(columns={"index":"pairs_num"},inplace=True)

In [120]:
df = df.reset_index()

In [125]:
for col in df.columns:
    df[col] = df[col].astype(float)

In [132]:
df =df.sort_values(["pairs_num", "index"])

In [134]:
i = 0
for index, row in df.iterrows():
    df.at[index, "iteration"] = int(i+1)
    i=i+1
    if i%10 == 0:
        i=0

In [138]:
df.head(20)

,iteration,pairs_num,AStar_nx,AStar_ride,AStar_ride_sub,BiDijkstra_ride,Dijkstra_nx,Dijkstra_ride,Dijkstra_ride_sub,OSMNX
0,1.0,0.0,0.000706,0.000024,0.000023,0.000035,0.001006,0.000015,0.000014,2.319543
1002,2.0,0.0,0.000165,0.000042,0.000074,0.000060,0.000073,0.000032,0.000053,5.978001
2000,3.0,0.0,0.000044,0.000013,0.000013,0.000026,0.000020,0.000009,0.000009,3.684128
3001,4.0,0.0,0.000450,0.002094,0.000203,0.001005,0.000183,0.001541,0.000166,3.799364
4002,5.0,0.0,0.000197,0.000061,0.000109,0.000098,0.000101,0.000049,0.000088,5.278305
5001,6.0,0.0,0.000702,0.002548,0.000243,0.001181,0.000386,0.002038,0.000173,3.359348
6002,7.0,0.0,0.000248,0.000041,0.000069,0.000060,0.000074,0.000032,0.000062,4.486967
7001,8.0,0.0,0.000684,0.002723,0.000206,0.001002,0.000432,0.001781,0.000172,3.028844
8002,9.0,0.0,0.000143,0.000040,0.000079,0.000061,0.000072,0.000032,0.000181,4.732949
9000,10.0,0.0,0.000047,0.000014,0.000013,0.000027,0.000020,0.000009,0.000010,2.575489


In [137]:
df.drop(columns=["index"], inplace=True)

In [78]:
# Функция для парсинга значений
for key, values in tqdm(data.items()):
    print(key)
    i = 0
    for algo in key:
    for entry in values:
        # print(entry)
        index, time = entry.split(":")
        # df.at[i, "index"] = index
        # df.at[i, key] = float(time)
        i = len(df)+1
        break

100%|██████████| 8/8 [00:00<?, ?it/s]

AStar_nx
Dijkstra_nx
AStar_ride
Dijkstra_ride
BiDijkstra_ride
AStar_ride_sub
Dijkstra_ride_sub
OSMNX


In [144]:
df.describe()

,iteration,pairs_num,AStar_nx,AStar_ride,AStar_ride_sub,BiDijkstra_ride,Dijkstra_nx,Dijkstra_ride,Dijkstra_ride_sub,OSMNX
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,5.500000,499.500000,4.878977,2.924853,0.179879,2.735312,4.876187,2.935300,0.213226,14.463280
std,2.872425,288.689425,6.602988,4.103072,0.447502,4.328793,6.660824,4.105070,0.518172,9.409759
min,1.000000,0.000000,0.000022,0.000009,0.000010,0.000018,0.000014,0.000007,0.000008,1.723768
25%,3.000000,249.750000,0.000516,0.000446,0.000387,0.000247,0.000289,0.000336,0.000324,7.986639
50%,5.500000,499.500000,0.819095,0.431422,0.006559,0.130519,0.805358,0.437354,0.005373,11.052554
75%,8.000000,749.250000,8.937267,5.175563,0.133346,4.149296,8.745168,5.148264,0.178125,17.848868
max,10.000000,999.000000,36.620468,24.118791,7.881016,32.906248,39.593369,26.504721,9.029374,58.673593


In [152]:
df=df.sort_values(["iteration", "pairs_num"])

In [153]:
df.reset_index().drop(columns=["index"], inplace=True)

In [140]:
df.astype("str").to_excel("Результат_1000_str.xlsx", index=False)

In [156]:
df.describe().to_excel("Describe_1000.xlsx", index=False)

In [154]:
df

,iteration,pairs_num,AStar_nx,AStar_ride,AStar_ride_sub,BiDijkstra_ride,Dijkstra_nx,Dijkstra_ride,Dijkstra_ride_sub,OSMNX
0,1.0,0.0,0.000706,0.000024,0.000023,0.000035,0.001006,0.000015,0.000014,2.319543
2,1.0,1.0,0.000205,0.000063,0.000115,0.000107,0.000107,0.000046,0.000086,2.791575
1,1.0,2.0,0.000423,0.003224,0.000313,0.001481,0.000256,0.001965,0.000173,2.396084
4,1.0,3.0,0.000090,0.000126,0.000135,0.000072,0.000040,0.000091,0.000118,4.155887
5,1.0,4.0,0.000346,0.000187,0.000108,0.000054,0.000223,0.000080,0.000092,4.500525
...,...,...,...,...,...,...,...,...,...,...
9984,10.0,995.0,3.626897,9.915688,0.768754,5.834405,3.874235,14.060013,0.719272,10.963105
9994,10.0,996.0,10.862367,4.266957,0.072855,13.843574,11.352036,6.666522,0.449318,19.617662
9993,10.0,997.0,12.076272,3.038027,0.257264,6.576387,33.154744,11.256589,0.082495,21.382660
9992,10.0,998.0,8.537828,8.081960,0.032505,7.197149,7.993899,9.991258,0.077979,23.623122
